# COCO dataset analyser

## This app is the intro into coco annotation controller and analyser. 

We have exported COCO datasets (multiple) from Roboflow. We want to do some basic calculations and analytics on these datasets. For instance, how many images and examples of spillage at the seals do we have? And how many at the centre deck?

Why do we want to know? 

## Dataset Balancing
We are detecting a single type of object: Spillage. But spillage can be found in multiple circumstances that visually differ a lot. On an external floating roof, we mainly see two kinds. Spillage near the seal and spillage on roof support legs or other similar extruding parts. Spillage near the seal is coming from scraped off wax that falls in between the tank roof and the wall and causes blockage that shows up when it's raining. It has a distinct flowing pattern that we call a stream or a flow. The severity is less because it's not a direct leak, but it occurs often. Spillage coming from legs is more severe but occurs many times less often. This causes an imbalance in the dataset.

## Roboflow Tagging
We make a distinction between classes and tags in Roboflow. We target everything as spillage

## get analysis from single coco file
We want:
- spillage seal amount
- spillage non-seal amount 

- annotations per image (seal, non-seal, mixed)
- total annotations

In [1]:
from pathlib import Path

import dataset.coco_models as coco_models
import dataset.coco_utils as coco_utils

from dataset.coco_models import DataSetMeta
from dataset.coco_utils import compute_tag_stats, filter_images_by_tag

In [2]:
root = Path(
    r"C:\Users\Gebruiker\Documents\Falcker\AI\data\OlieDetectie\spillage_large_detection_exports\SpillageLargeDetection.v22-count.coco - kopie"
)
ds_meta = DataSetMeta.from_dir(root)
ds = ds_meta.merge_self()

stats = compute_tag_stats(ds)

total_images = ds_meta.image_count("all")
print(f"Total images: {total_images}")
print("Tag counts:", stats["tag_counts"])

images_with_wrong_names = filter_images_by_tag(ds_meta.train_COCO, "wrong-naming")


# print(images_with_wrong_names[0:10])
# 

Total images: 388
Tag counts: {'seal': 252, 'leg': 107, 'wrong-naming': 178, 'plate': 36, 'possible-spillage': 1, 'possible-oil': 1, 'vent': 1}


In [3]:
ds_meta.normalize_filenames()

In [4]:
ds_meta.all_COCO.images[0:10]

[COCOImage(id=0, file_name='F52_20260209.jpg', width=320, height=320, date_captured=datetime.datetime(2026, 2, 9, 14, 42, 51, tzinfo=TzInfo(UTC)), asset_name='F52', original_file_name='20250815-F52_jpeg.rf.b5b7acb24e882434ebf39322177a7cae.jpg', subset='train', extra=Extra(name='20250815-F52.jpeg', user_tags=['seal'])),
 COCOImage(id=1, file_name='F63_20260209.jpg', width=320, height=320, date_captured=datetime.datetime(2026, 2, 9, 14, 42, 51, tzinfo=TzInfo(UTC)), asset_name='F63', original_file_name='230706_T063_jpg.rf.e2a51fa3d029a7619361c600e209217d.jpg', subset='train', extra=Extra(name='230706_T063.jpg', user_tags=['seal', 'leg', 'wrong-naming'])),
 COCOImage(id=2, file_name='F09_20260209.jpg', width=320, height=320, date_captured=datetime.datetime(2026, 2, 9, 14, 42, 51, tzinfo=TzInfo(UTC)), asset_name='F09', original_file_name='20250620-F09_jpeg.rf.f6d68ce818e365992b82fa902b608038.jpg', subset='train', extra=Extra(name='20250620-F09.jpeg', user_tags=['seal'])),
 COCOImage(id=3, f

In [5]:
ds_meta.change_filenames_on_dir()

FileExistsError: [WinError 183] Kan geen bestand maken dat al bestaat: 'C:\\Users\\Gebruiker\\Documents\\Falcker\\AI\\data\\OlieDetectie\\spillage_large_detection_exports\\SpillageLargeDetection.v22-count.coco - kopie\\train\\20250901-F-04-Roof_jpeg.rf.7da013bc76524109af142e9c44049c44.jpg' -> 'C:\\Users\\Gebruiker\\Documents\\Falcker\\AI\\data\\OlieDetectie\\spillage_large_detection_exports\\SpillageLargeDetection.v22-count.coco - kopie\\train\\F04_20260209.jpg'

In [ ]:
coco_annotations_path = Path(
    r"C:\Users\Gebruiker\Documents\Falcker\AI\data\OlieDetectie\spillage_large_detection_exports\SpillageLargeDetection.v19-v1.12.coco\test\_annotations.coco.json"
)
root = Path(
    r"C:\Users\Gebruiker\Documents\Falcker\AI\data\OlieDetectie\spillage_large_detection_exports\SpillageLargeDetection.v22-count.coco"
)
ds = DataSetMeta.from_dir(root)

print(ds.summary())

print("Train images:", ds.image_count("train"))
print("Valid images:", ds.image_count("valid"))
print("Test images:", ds.image_count("test"))

print("Category distribution (train):")
print(ds.category_counts("train"))
coco = COCO(coco_annotations_path)
# Show all category names
cats = coco.loadCats(coco.getCatIds())
print([c["name"] for c in cats])

img = ds.train_coco.get_image(1)

images = coco.loadImgs

# Get all annotation IDs for one category
cat_id = coco.getCatIds(catNms=["Spillage"])
ann_ids = coco.getAnnIds(catIds=cat_id)
annotations = coco.loadAnns(ann_ids)

print(annotations)

annotations_list = []


for ann in annotations[0:10]:
    print(ann)
    new_coco = CocoAnnotation(**ann)
    annotations_list.append(new_coco)
    # ann["category_id"]


def read_coco(path: Path) -> COCO:
    return COCO(path)
